# 04 — QAOA Generalization and Held-Out Noise Benchmark

**QuantumOptLab | Phase 4 | Seed 31 | Qiskit 2.6.2 + Aer 0.18.2**

This notebook expands the three-instance pilot into a **predeclared graph benchmark**. It implements exact Max-Cut, greedy search, simulated annealing, QAOA depths $p=1$ and $p=2$, and a factorial subset of independent CX/readout noise conditions.

**Important experimental boundary:** 80 connected graph instances are generated with fixed seeds; 40 are designated training, 20 validation, and **20 locked test**. Only training/validation instances are optimized. The locked test set receives **no QAOA or noise outcome labels** in Phase 4, preserving it for later, pre-registered final evaluation. The validation set is held out from our earlier three-graph pilot, but it is *not* a blind final test of a subsequently developed selector.

Parameter optimization uses an exact NumPy QAOA amplitude implementation, validated against Qiskit's `Statevector`. Circuit compilation and noisy sampling use Qiskit/Aer. None of these experiments establish quantum advantage or real-hardware performance.

Use **Run All** in the `Python (QuantumOptLab)` kernel. Markdown explanations immediately precede the code that implements each step. No project-local Python imports are required.

### 1. Configure the reproducible experiment

The full benchmark generates 80 graphs and optimizes 60 (training + validation). For a quicker end-to-end check, set `RUN_MODE = "smoke"` below, then return to `"full"` for reportable results. The smoke run uses different output names and is **not** research evidence. Saved checkpoint rows are reused only when the experiment configuration matches.

In [ ]:
from __future__ import annotations

from pathlib import Path
from hashlib import sha256
import json
import math
import platform
import time

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from IPython.display import display
import qiskit
import qiskit_aer
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, ReadoutError, depolarizing_error

SEED = 31
RUN_MODE = "full"  # "smoke" for a much smaller functional check
assert RUN_MODE in ("full", "smoke")
SIZES = (6, 7, 8, 9, 10)
FAMILIES = ("er_sparse", "er_dense", "small_world", "weighted_er")
DEPTHS = (1, 2)
RESTARTS = 2 if RUN_MODE == "full" else 1
MAXITER = 55 if RUN_MODE == "full" else 12
SHOTS = 1024 if RUN_MODE == "full" else 512
NOISE_REPEATS = 3 if RUN_MODE == "full" else 1
NOISE_CONDITIONS = (
    ("ideal", 0.0, 0.0),
    ("cx_002", 0.02, 0.0),
    ("cx_005", 0.05, 0.0),
    ("cx_010", 0.10, 0.0),
    ("readout_002", 0.0, 0.02),
    ("cx_005_readout_002", 0.05, 0.02),
)

ROOT = Path.cwd()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "notebooks").exists(), f"Open this notebook inside QuantumOptLab: {ROOT}"
EXPERIMENTS = ROOT / "reports" / "experiments"
FIGURES = ROOT / "reports" / "figures"
EXPERIMENTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
PREFIX = f"phase04_{RUN_MODE}_seed{SEED}"
SETTINGS = {
    "seed": SEED, "run_mode": RUN_MODE, "sizes": SIZES, "families": FAMILIES,
    "depths": DEPTHS, "restarts": RESTARTS, "maxiter": MAXITER,
    "shots": SHOTS, "noise_repeats": NOISE_REPEATS,
    "noise_conditions": NOISE_CONDITIONS, "optimizer": "L-BFGS-B",
    "simulator": "Aer density_matrix", "basis_gates": ["h", "rx", "rz", "cx"],
    "transpile_optimization_level": 0, "method_version": "qaoa-exact-vector-v1",
}
CONFIG_HASH = sha256(json.dumps(SETTINGS, sort_keys=True).encode()).hexdigest()[:12]
RUN_START = time.perf_counter()
print("Project:", ROOT)
print("Qiskit:", qiskit.__version__, "| Aer:", qiskit_aer.__version__)
print("Mode:", RUN_MODE, "| Hash:", CONFIG_HASH, "| Seed:", SEED)
print("Noisy validation samples:", len(NOISE_CONDITIONS)*2*20*NOISE_REPEATS if RUN_MODE == "full" else "smoke subset")


### 2. Generate and freeze the graph suite and splits

Create four graph families at 6–10 vertices, with four instances per size/family stratum. Reject disconnected graphs and duplicate **Weisfeiler–Lehman graph hashes** (with edge weights), a conservative safeguard against some structural duplication. Assign two instances per stratum to training, one to validation, and one to locked test using a fixed pseudorandom permutation. A WL hash is not a complete graph-isomorphism proof, so we also run pairwise isomorphism checks between the active splits and locked test graphs.

In [ ]:
def construct_graph(family, n, seed):
    rng = np.random.default_rng(seed)
    if family == "er_sparse":
        g = nx.erdos_renyi_graph(n, 0.38, seed=seed)
    elif family == "er_dense":
        g = nx.erdos_renyi_graph(n, 0.64, seed=seed)
    elif family == "small_world":
        g = nx.watts_strogatz_graph(n, 4, 0.55, seed=seed)
    elif family == "weighted_er":
        g = nx.erdos_renyi_graph(n, 0.50, seed=seed)
    else:
        raise ValueError(family)
    if not nx.is_connected(g):
        return None
    for u, v in g.edges():
        g[u][v]["weight"] = int(rng.integers(1, 4)) if family == "weighted_er" else 1
    return nx.convert_node_labels_to_integers(g, ordering="sorted")

catalog = []
graphs = {}
seen_hashes = set()
for n in SIZES:
    for family_idx, family in enumerate(FAMILIES):
        shuffle_rng = np.random.default_rng(SEED + 90000 + 100*n + family_idx)
        split_slots = ["train", "train", "validation", "test"]
        shuffle_rng.shuffle(split_slots)
        for replicate in range(4):
            graph_id = f"n{n}_{family}_r{replicate+1:02d}"
            for attempt in range(2500):
                graph_seed = SEED + n*1_000_000 + family_idx*50_000 + replicate*2500 + attempt
                graph = construct_graph(family, n, graph_seed)
                if graph is None:
                    continue
                fingerprint = nx.weisfeiler_lehman_graph_hash(graph, edge_attr="weight")
                if fingerprint not in seen_hashes:
                    break
            else:
                raise RuntimeError(f"Insufficient distinct connected graphs for {graph_id}")
            seen_hashes.add(fingerprint)
            graphs[graph_id] = graph
            catalog.append({
                "graph_id": graph_id, "n": n, "family": family,
                "split": split_slots[replicate], "graph_seed": graph_seed,
                "fingerprint": fingerprint, "edges": graph.number_of_edges(),
                "weight_sum": sum(float(d["weight"]) for *_, d in graph.edges(data=True)),
                "avg_degree": 2*graph.number_of_edges()/n,
                "clustering": nx.average_clustering(graph),
            })

catalog_df = pd.DataFrame(catalog).sort_values("graph_id").reset_index(drop=True)
assert len(catalog_df) == 80 and catalog_df.graph_id.is_unique
assert catalog_df.groupby(["n", "family", "split"]).size().unstack(fill_value=0).eq(
    pd.Series({"train": 2, "validation": 1, "test": 1})
).all().all(), "Split balance changed"
assert catalog_df.fingerprint.is_unique
# Isomorphism screening prevents common small-graph structures leaking from train/validation to test.
active_graphs = [graphs[id_] for id_ in catalog_df.loc[catalog_df.split != "test", "graph_id"]]
locked_graphs = [graphs[id_] for id_ in catalog_df.loc[catalog_df.split == "test", "graph_id"]]
for left in active_graphs:
    for right in locked_graphs:
        if left.number_of_nodes() == right.number_of_nodes() and left.number_of_edges() == right.number_of_edges():
            assert not nx.is_isomorphic(
                left, right, edge_match=lambda a, b: a["weight"] == b["weight"]
            ), "Graph isomorphism across the locked-test boundary"
ACTIVE_IDS = catalog_df.loc[catalog_df.split != "test", "graph_id"].tolist()
VALIDATION_IDS = catalog_df.loc[catalog_df.split == "validation", "graph_id"].tolist()
if RUN_MODE == "smoke":
    train_sample = catalog_df.loc[catalog_df.split == "train"].groupby("family", sort=True).head(1).graph_id.tolist()
    validation_sample = catalog_df.loc[catalog_df.split == "validation"].groupby("family", sort=True).head(1).graph_id.tolist()
    ACTIVE_IDS = sorted(train_sample + validation_sample)
    VALIDATION_IDS = sorted(validation_sample)

catalog_path = EXPERIMENTS / f"{PREFIX}_graph_catalog.json"
catalog_path.write_text(json.dumps({
    "seed": SEED, "config_hash": CONFIG_HASH, "records": [
        {**row, "edge_list": [(int(u), int(v), int(d["weight"]))
            for u, v, d in graphs[row["graph_id"]].edges(data=True)]}
        for row in catalog_df.to_dict("records")
    ], "test_policy": "Do not optimize or inspect test outcomes until final pre-registered evaluation"
}, indent=2), encoding="utf-8")
display(catalog_df.groupby(["n", "family", "split"]).size().unstack(fill_value=0))
print("Generated", len(graphs), "graphs | active:", len(ACTIVE_IDS), "| validation noisy:", len(VALIDATION_IDS))
print("Saved graph catalog:", catalog_path)


### 3. Establish exact Max-Cut solutions and classical heuristics

Exact enumeration is feasible through 10 vertices and supplies a trustworthy reference. Greedy single-bit-flip local search and simulated annealing are **classical heuristic baselines**. For both, only their best sampled cut is recorded; QAOA instead reports *expected* cut and probability of sampling an optimal cut. These different metrics are intentionally **not** interpreted as a speed or quantum-advantage contest. We also record objective-function evaluation counts.

In [ ]:
def cut_value(g, x):
    return float(sum(d["weight"] for u, v, d in g.edges(data=True)
                     if ((x >> u) & 1) != ((x >> v) & 1)))


def exact_cost_table(g):
    n = g.number_of_nodes()
    assignment = np.arange(1 << n, dtype=np.uint64)
    costs = np.zeros(1 << n, dtype=np.float64)
    for u, v, data in g.edges(data=True):
        costs += float(data["weight"]) * (((assignment >> u) ^ (assignment >> v)) & 1)
    return costs


def greedy_search(g, seed, starts=20):
    rng = np.random.default_rng(seed)
    best = -float("inf")
    calls = 0
    n = g.number_of_nodes()
    for _ in range(starts):
        x = int(rng.integers(0, 1 << n))
        current = cut_value(g, x)
        calls += 1
        while True:
            candidate_scores = [(cut_value(g, x ^ (1 << j)), j) for j in range(n)]
            calls += n
            candidate, bit = max(candidate_scores, key=lambda t: t[0])
            if candidate <= current + 1e-12:
                break
            x ^= (1 << bit)
            current = candidate
        best = max(best, current)
    return float(best), calls


def simulated_annealing(g, seed, restarts=12, steps=160):
    rng = np.random.default_rng(seed)
    n = g.number_of_nodes()
    best = -float("inf")
    calls = 0
    scale = max(1., sum(d["weight"] for *_, d in g.edges(data=True))/n)
    for _ in range(restarts):
        x = int(rng.integers(0, 1 << n))
        current = cut_value(g, x)
        calls += 1
        best = max(best, current)
        for step in range(steps):
            temperature = max(.05*scale, 1.6*scale * (0.98**step))
            candidate_x = x ^ (1 << int(rng.integers(n)))
            candidate = cut_value(g, candidate_x)
            calls += 1
            if candidate >= current or rng.random() < np.exp((candidate-current)/temperature):
                x, current = candidate_x, candidate
                best = max(best, current)
    return float(best), calls

COSTS, OPTIMA, CLASSICAL_ROWS = {}, {}, []
for position, graph_id in enumerate(ACTIVE_IDS):
    g = graphs[graph_id]
    costs = exact_cost_table(g)
    optimum = float(costs.max())
    assert optimum > 0
    # Compare the vectorized classical objective against direct edge enumeration.
    assert all(abs(costs[x]-cut_value(g, x)) < 1e-10 for x in range(len(costs)))
    greedy, greedy_calls = greedy_search(g, SEED+position*97)
    anneal, anneal_calls = simulated_annealing(g, SEED+position*97+1)
    assert greedy <= optimum and anneal <= optimum
    COSTS[graph_id], OPTIMA[graph_id] = costs, optimum
    detail = catalog_df.set_index("graph_id").loc[graph_id]
    CLASSICAL_ROWS.append({
        "graph_id": graph_id, "split": detail["split"], "family": detail["family"],
        "n": int(detail["n"]), "optimum": optimum,
        "n_optimal_bitstrings": int(np.isclose(costs, optimum).sum()),
        "uniform_expected_cut": float(costs.mean()),
        "greedy_best_cut": greedy, "greedy_ratio": greedy/optimum, "greedy_calls": greedy_calls,
        "anneal_best_cut": anneal, "anneal_ratio": anneal/optimum, "anneal_calls": anneal_calls,
    })
classical_df = pd.DataFrame(CLASSICAL_ROWS)
classical_df.to_csv(EXPERIMENTS / f"{PREFIX}_classical.csv", index=False)
display(classical_df.groupby("split")[["greedy_ratio", "anneal_ratio"]].agg(["mean", "min"]).round(4))
print("Exact enumeration and both classical methods complete for", len(COSTS), "active graphs")


### 4. Implement accelerated exact QAOA and prove its equivalence to Qiskit

For each layer, apply $e^{-i\gamma C(z)}$ to every computational-basis amplitude, followed by $e^{-i\beta X}$ on each qubit. This is the same QAOA convention as Phase 2, including `rzz(-gamma * edge_weight)` and `rx(2 * beta)`.

The NumPy amplitude evaluator makes per-instance optimization substantially more practical. **Before trusting it**, verify QAOA probabilities against `Statevector.from_instruction` on several graphs and both circuit depths. Qiskit remains the authoritative circuit representation for transpilation and Aer noise experiments.

In [ ]:
def qaoa_circuit(g, params, depth):
    params = np.asarray(params, dtype=float)
    assert params.shape == (2*depth,)
    qc = QuantumCircuit(g.number_of_nodes())
    qc.h(range(g.number_of_nodes()))
    for gamma, beta in zip(params[:depth], params[depth:]):
        for u, v, data in g.edges(data=True):
            qc.rzz(-float(gamma) * float(data["weight"]), u, v)
        for q in range(g.number_of_nodes()):
            qc.rx(2*float(beta), q)
    return qc


def fast_qaoa_probabilities(n, costs, params, depth):
    params = np.asarray(params, dtype=float)
    amps = np.full(1 << n, 1./np.sqrt(1 << n), dtype=np.complex128)
    positions = np.arange(1 << n, dtype=np.uint64)
    pairs = []
    for bit in range(n):
        left = np.flatnonzero(((positions >> bit) & 1) == 0)
        pairs.append((left, left | (1 << bit)))
    for gamma, beta in zip(params[:depth], params[depth:]):
        amps *= np.exp(-1j * float(gamma) * costs)
        c, s = np.cos(beta), -1j*np.sin(beta)
        for idx0, idx1 in pairs:
            left, right = amps[idx0].copy(), amps[idx1].copy()
            amps[idx0] = c*left + s*right
            amps[idx1] = s*left + c*right
    probs = np.abs(amps)**2
    return probs


def qaoa_stats(graph_id, params, depth):
    probs = fast_qaoa_probabilities(graphs[graph_id].number_of_nodes(), COSTS[graph_id], params, depth)
    costs = COSTS[graph_id]
    return (float(probs @ costs), float(probs[np.isclose(costs, OPTIMA[graph_id])].sum()))

rng_check = np.random.default_rng(SEED + 403)
physics_cases = [ACTIVE_IDS[0], ACTIVE_IDS[len(ACTIVE_IDS)//2], ACTIVE_IDS[-1]]
max_prob_error = 0.0
for graph_id in physics_cases:
    for depth in DEPTHS:
        params = np.concatenate([rng_check.uniform(0,2*np.pi,depth),rng_check.uniform(0,np.pi,depth)])
        reference = np.abs(Statevector.from_instruction(qaoa_circuit(graphs[graph_id],params,depth)).data)**2
        candidate = fast_qaoa_probabilities(graphs[graph_id].number_of_nodes(),COSTS[graph_id],params,depth)
        max_prob_error = max(max_prob_error, float(np.max(np.abs(reference-candidate))))
assert max_prob_error < 1e-10, f"QAOA implementation disagrees with Qiskit: {max_prob_error}"
print(f"PASS: Fast exact QAOA agrees with Qiskit Statevector; max probability error={max_prob_error:.2e}")


### 5. Optimize QAOA depths 1 and 2 on training/validation graphs

Use fixed-angle bounds, two independent restarts per instance/depth (one in smoke mode), and L-BFGS-B. Each optimizer restart is recorded separately, including its parameters, function evaluations and convergence flag. Results are checkpointed after every restart so an interrupted long run can resume. **No test graph is optimized.** A saved checkpoint with a different configuration is rejected.

In [ ]:
TRIAL_FILE = EXPERIMENTS / f"{PREFIX}_optimizer_restarts.csv"
if TRIAL_FILE.exists():
    trial_rows = pd.read_csv(TRIAL_FILE).to_dict("records")
    assert all(row["config_hash"] == CONFIG_HASH for row in trial_rows), (
        "Configuration changed since checkpoint. Rename/remove old phase04 optimizer checkpoint first."
    )
else:
    trial_rows = []
completed_trials = {(row["graph_id"], int(row["depth_p"]), int(row["restart"])) for row in trial_rows}
optimization_start = time.perf_counter()
for graph_position, graph_id in enumerate(ACTIVE_IDS):
    for depth in DEPTHS:
        for restart in range(RESTARTS):
            key = (graph_id, depth, restart)
            if key in completed_trials:
                continue
            opt_seed = SEED + 100_000*graph_position + 1000*depth + restart
            generator = np.random.default_rng(opt_seed)
            init = np.concatenate([
                generator.uniform(0, 2*np.pi, size=depth),
                generator.uniform(0, np.pi, size=depth),
            ])
            def negative_expected_cut(params):
                return -qaoa_stats(graph_id,params,depth)[0]
            result = minimize(
                negative_expected_cut, init, method="L-BFGS-B",
                bounds=[(0,2*np.pi)]*depth + [(0,np.pi)]*depth,
                options={"maxiter": MAXITER, "maxfun": 600, "ftol": 1e-9},
            )
            mean_cut, prob_optimum = qaoa_stats(graph_id,result.x,depth)
            trial_rows.append({
                "config_hash": CONFIG_HASH, "graph_id": graph_id,
                "depth_p": depth, "restart": restart, "optimizer_seed": opt_seed,
                "expected_cut": mean_cut, "p_optimal": prob_optimum,
                "approx_ratio": mean_cut / OPTIMA[graph_id],
                "params_json": json.dumps([float(x) for x in result.x]),
                "nfev": int(result.nfev), "nit": int(result.nit),
                "converged": bool(result.success),
            })
            pd.DataFrame(trial_rows).to_csv(TRIAL_FILE, index=False)
            completed_trials.add(key)
    if (graph_position+1)%10 == 0:
        print(f"Optimized {graph_position+1}/{len(ACTIVE_IDS)} graph instances")

trial_df = pd.DataFrame(trial_rows)
assert len(trial_df) == len(ACTIVE_IDS)*len(DEPTHS)*RESTARTS
trial_df = trial_df.sort_values("expected_cut",ascending=False)
best_df = trial_df.drop_duplicates(["graph_id","depth_p"]).copy()
best_df = best_df.merge(catalog_df[["graph_id","split","family","n","edges","weight_sum"]],on="graph_id",validate="many_to_one")
best_df["optimum"] = best_df.graph_id.map(OPTIMA)
best_df = best_df.sort_values(["graph_id","depth_p"]).reset_index(drop=True)
best_df.to_csv(EXPERIMENTS / f"{PREFIX}_qaoa_ideal.csv",index=False)
print("Saved",len(trial_df),"optimization trials and",len(best_df),"selected depth results")
print("Optimizer elapsed (including checkpoint reuses):",round(time.perf_counter()-optimization_start,1),"seconds")
display(best_df.groupby(["split","depth_p"])[["approx_ratio","p_optimal"]].mean().round(4))


### 6. Compile the validation circuits with CX gates and record circuit resources

Compile with the same CX-based gate set and optimization level 0 as Phase 3. Store CX counts and circuit depth. This is a deliberately controlled noise comparison; it does not claim to represent device-specific routing or optimized hardware compilation. Only validation circuits are prepared for noise sampling.

In [ ]:
COMPILED = {}
resource_rows = []
for graph_id in VALIDATION_IDS:
    g = graphs[graph_id]
    for depth in DEPTHS:
        row = best_df.loc[(best_df.graph_id == graph_id)&(best_df.depth_p == depth)].iloc[0]
        params = np.asarray(json.loads(row.params_json),dtype=float)
        qc = qaoa_circuit(g,params,depth)
        compiled = transpile(qc,basis_gates=["h","rx","rz","cx"],
                            optimization_level=0,seed_transpiler=SEED)
        operations = dict(compiled.count_ops())
        assert "rzz" not in operations
        count_cx = int(operations.get("cx",0))
        assert count_cx == 2*g.number_of_edges()*depth
        compiled.measure_all()
        COMPILED[graph_id,depth] = compiled
        resource_rows.append({"graph_id":graph_id,"depth_p":depth,"cx_gates":count_cx,
            "compiled_depth_with_measure":compiled.depth(),"total_operations":int(sum(compiled.count_ops().values()))})
resource_df = pd.DataFrame(resource_rows)
resource_df.to_csv(EXPERIMENTS / f"{PREFIX}_circuit_resources.csv", index=False)
display(resource_df.groupby("depth_p")[["cx_gates","compiled_depth_with_measure"]].agg(["mean","min","max"]).round(2))


### 7. Evaluate *independent* CX and readout noise conditions with Aer

The independent settings below include CX-only and readout-only controls, as well as a combined condition. The noise model applies a two-qubit depolarizing error **after CX** and/or independent symmetric readout bit flips. The parameters are synthetic and not IBM hardware calibration data. All validation circuits retain their **original ideal-trained angles**; no noise re-optimization is performed. Each condition is repeated with separate simulator seeds.

Aer runs batches of circuits to reduce job overhead, saving a CSV checkpoint after every small batch. On ordinary laptop hardware, this is the most computationally intensive notebook section.

In [ ]:
def make_noise_model(cx_p, readout_p):
    model = NoiseModel()
    if cx_p:
        model.add_all_qubit_quantum_error(depolarizing_error(float(cx_p),2),["cx"])
    if readout_p:
        q = float(readout_p)
        model.add_all_qubit_readout_error(ReadoutError([[1-q,q],[q,1-q]]))
    return model


def summarize_counts(graph_id,counts):
    costs = COSTS[graph_id]
    optimum = OPTIMA[graph_id]
    shots = sum(int(n) for n in counts.values())
    cut_sum = 0.0
    optimum_count = 0
    for bits, count in counts.items():
        assignment = int(bits.replace(" ",""),2)
        value = costs[assignment]
        cut_sum += value*int(count)
        if np.isclose(value,optimum):
            optimum_count += int(count)
    return float(cut_sum/shots),float(optimum_count/shots),shots

NOISE_RAW_PATH = EXPERIMENTS / f"{PREFIX}_noise_raw.csv"
if NOISE_RAW_PATH.exists():
    noise_rows = pd.read_csv(NOISE_RAW_PATH).to_dict("records")
    assert all(row["config_hash"] == CONFIG_HASH for row in noise_rows), (
        "Configuration changed since noise checkpoint. Rename/remove old checkpoint first."
    )
else:
    noise_rows = []
completed_noise = {(row["graph_id"],int(row["depth_p"]),row["condition"],int(row["repeat"]))
                    for row in noise_rows}
BATCH_SIZE = 8
noise_start = time.perf_counter()
for condition_index, (condition_name,cx_p,readout_p) in enumerate(NOISE_CONDITIONS):
    simulator = AerSimulator(method="density_matrix",noise_model=make_noise_model(cx_p,readout_p),
                            max_parallel_experiments=1)
    for repeat in range(NOISE_REPEATS):
        pending = [(graph_id,depth) for graph_id in VALIDATION_IDS for depth in DEPTHS
                   if (graph_id,depth,condition_name,repeat) not in completed_noise]
        for offset in range(0,len(pending),BATCH_SIZE):
            batch = pending[offset:offset+BATCH_SIZE]
            circuits = [COMPILED[key] for key in batch]
            simulator_seed = SEED + 100_000*condition_index + 1000*repeat + offset
            result = simulator.run(circuits,shots=SHOTS,seed_simulator=simulator_seed).result()
            assert result.success, f"Aer simulation failed for {condition_name}"
            for circuit_index,(graph_id,depth) in enumerate(batch):
                counts = result.get_counts(circuit_index)
                mean_cut,p_optimal,actual_shots = summarize_counts(graph_id,counts)
                assert actual_shots == SHOTS
                noise_rows.append({
                    "config_hash":CONFIG_HASH,"graph_id":graph_id,"depth_p":depth,
                    "condition":condition_name,"cx_p":cx_p,"readout_p":readout_p,
                    "repeat":repeat,"simulator_seed":simulator_seed,"shots":actual_shots,
                    "expected_cut_shots":mean_cut,"approx_ratio":mean_cut/OPTIMA[graph_id],
                    "p_optimal":p_optimal,
                })
                completed_noise.add((graph_id,depth,condition_name,repeat))
            pd.DataFrame(noise_rows).to_csv(NOISE_RAW_PATH,index=False)
    print(f"Finished {condition_name}: {len(VALIDATION_IDS)*len(DEPTHS)*NOISE_REPEATS} records")

noise_df = pd.DataFrame(noise_rows)
expected_noise_rows = len(VALIDATION_IDS)*len(DEPTHS)*len(NOISE_CONDITIONS)*NOISE_REPEATS
assert len(noise_df)==expected_noise_rows
assert not noise_df.duplicated(["graph_id","depth_p","condition","repeat"]).any()
print("Noise simulations:",len(noise_df),"| Noise section elapsed:",round(time.perf_counter()-noise_start,1),"seconds")


### 8. Aggregate paired observations and estimate between-graph uncertainty

Average repetitions *within* each graph/condition/depth before comparing depths. The paired difference in approximation ratio is `p=2 minus p=1`. Bootstrap whole **graph instances** (not individual shots) to estimate uncertainty in the mean depth benefit over these 20 validation graphs. These are descriptive percentile confidence intervals conditional on the fixed graph suite, ideal-trained parameters, and simulator noise model, not proofs of generalization or quantum advantage.

In [ ]:
noise_summary = noise_df.groupby(["graph_id","depth_p","condition","cx_p","readout_p"],as_index=False).agg(
    mean_ratio=("approx_ratio","mean"),std_ratio=("approx_ratio","std"),
    mean_p_optimal=("p_optimal","mean"),n_repeats=("repeat","count")
)
noise_summary = noise_summary.merge(catalog_df[["graph_id","family","n","split"]],on="graph_id",validate="many_to_one")
noise_summary.to_csv(EXPERIMENTS / f"{PREFIX}_noise_summary.csv",index=False)

paired = noise_summary.pivot_table(index=["graph_id","family","n","condition","cx_p","readout_p"],
                                    columns="depth_p",values="mean_ratio",aggfunc="first").reset_index()
paired.columns = [f"p{int(c)}" if isinstance(c,(int,np.integer)) else c for c in paired.columns]
assert {"p1","p2"}.issubset(paired.columns)
paired["depth2_minus_depth1"] = paired.p2-paired.p1
paired.to_csv(EXPERIMENTS / f"{PREFIX}_paired_depth_effects.csv",index=False)

def bootstrap_mean_interval(values,seed=SEED,n_boot=4000):
    values = np.asarray(values,dtype=float)
    rng = np.random.default_rng(seed)
    draws = rng.choice(values,size=(n_boot,len(values)),replace=True).mean(axis=1)
    return float(values.mean()),float(np.quantile(draws,.025)),float(np.quantile(draws,.975))

condition_rows=[]
for condition_index,(condition_name,cx_p,readout_p) in enumerate(NOISE_CONDITIONS):
    data=paired[paired.condition==condition_name]
    mu,lo,hi=bootstrap_mean_interval(data.depth2_minus_depth1,seed=SEED+condition_index)
    condition_rows.append({"condition":condition_name,"cx_p":cx_p,"readout_p":readout_p,
                           "mean_depth_benefit":mu,"ci_95_low":lo,"ci_95_high":hi,
                           "n_graphs":len(data),"fraction_depth2_better":float((data.depth2_minus_depth1>0).mean())})
condition_df=pd.DataFrame(condition_rows)
condition_df.to_csv(EXPERIMENTS / f"{PREFIX}_depth_benefit_bootstrap.csv",index=False)
display(condition_df.round(4))


### 9. Visualize ideal-optimization performance across graph families

Show validation-only QAOA approximation ratios by graph family, and the per-graph ideal depth advantage. The train/validation separation is visible in the saved tables. Classical heuristic ratios are shown separately from QAOA expectation ratios, since a best cut and an expected cut answer different questions.

In [ ]:
val_ideal = best_df[best_df.split=="validation"].copy()
fig,axes=plt.subplots(1,2,figsize=(10.8,3.9))
by_family=val_ideal.groupby(["family","depth_p"])["approx_ratio"].mean().unstack()
by_family.plot(kind="bar",ax=axes[0],width=.75,rot=15)
axes[0].set_title("Validation QAOA Approximation Ratio")
axes[0].set_ylabel("Expected cut / exact maximum")
axes[0].set_xlabel("Graph family")
axes[0].set_ylim(0,1.04)
axes[0].legend(["p=1","p=2"],loc="lower right",fontsize=8)
wide=val_ideal.pivot(index="graph_id",columns="depth_p",values="approx_ratio")
ideal_delta=wide[2]-wide[1]
axes[1].hist(ideal_delta,bins=12,edgecolor="white")
axes[1].axvline(0,color="gray",linestyle="--",linewidth=1)
axes[1].set_title("Ideal Depth Benefit on Validation Graphs")
axes[1].set_xlabel("Approx. ratio difference (p=2 minus p=1)")
axes[1].set_ylabel("Graph count")
for ax in axes: ax.grid(axis="y",alpha=.2)
fig.tight_layout()
ideal_figure=FIGURES / f"{PREFIX}_ideal_generalization.png"
fig.savefig(ideal_figure,dpi=175,bbox_inches="tight")
plt.show()
print("Saved:",ideal_figure)
print("Validation exact heuristic best-of-run ratios:")
display(classical_df[classical_df.split=="validation"][["greedy_ratio","anneal_ratio"]].describe().round(4))


### 10. Visualize the noise/depth tradeoff and its uncertainty

Plot the mean paired depth benefit on validation graphs with graph-level bootstrap intervals. The CX-only conditions form a controlled x-axis sweep, while the readout-only and combined cases remain distinct comparisons. A crossed interval containing zero means we should avoid a categorical claim that one depth is reliably better under that condition.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(10.6,3.9))
cx_only=condition_df[condition_df.readout_p==0].sort_values("cx_p")
axes[0].errorbar(cx_only.cx_p,cx_only.mean_depth_benefit,
                 yerr=[cx_only.mean_depth_benefit-cx_only.ci_95_low,
                       cx_only.ci_95_high-cx_only.mean_depth_benefit],
                 marker="o",capsize=4)
axes[0].axhline(0,color="gray",linestyle="--",linewidth=1)
axes[0].set_title("CX Noise: Depth-2 Benefit")
axes[0].set_xlabel("CX depolarizing parameter")
axes[0].set_ylabel("Mean Δ approximation ratio (p=2 − p=1)")
axes[0].grid(alpha=.2)

condition_order=condition_df.sort_values(["cx_p","readout_p"]).reset_index(drop=True)
x=np.arange(len(condition_order))
axes[1].bar(x,condition_order.mean_depth_benefit)
axes[1].errorbar(x,condition_order.mean_depth_benefit,
                 yerr=[condition_order.mean_depth_benefit-condition_order.ci_95_low,
                       condition_order.ci_95_high-condition_order.mean_depth_benefit],
                 fmt="none",ecolor="black",capsize=3)
axes[1].set_xticks(x,condition_order.condition,rotation=42,ha="right",fontsize=8)
axes[1].axhline(0,color="gray",linestyle="--",linewidth=1)
axes[1].set_title("Depth Benefit by Noise Condition")
axes[1].set_ylabel("Mean Δ approximation ratio")
axes[1].grid(axis="y",alpha=.2)
fig.tight_layout()
noise_figure=FIGURES / f"{PREFIX}_depth_benefit_noise.png"
fig.savefig(noise_figure,dpi=175,bbox_inches="tight")
plt.show()
print("Saved:",noise_figure)


### 11. Verify experimental integrity and save the reproducibility manifest

Checks cover unique balanced graphs, Qiskit/NumPy physics equivalence, exact baselines, complete restart records, test-set isolation, CX decomposition, noise-repeat completeness, probability bounds, and the noise-free sampling control. The experiment manifest records the versions, seeds, settings, output paths, and elapsed time. A **PASS** here means the implementation ran consistently; it does not validate the scientific hypothesis by itself.

In [ ]:
ideal_noise=noise_summary[noise_summary.condition=="ideal"].merge(
    best_df[["graph_id","depth_p","approx_ratio"]],on=["graph_id","depth_p"],validate="one_to_one"
)
noise_control_mean_abs_error=float((ideal_noise.mean_ratio-ideal_noise.approx_ratio).abs().mean())
locked_ids=set(catalog_df.loc[catalog_df.split=="test","graph_id"])
checks={
    "80 distinct balanced graphs":len(catalog_df)==80 and catalog_df.fingerprint.is_unique,
    "Physics agrees with Qiskit":max_prob_error<1e-10,
    "Exact baselines cover active suite":len(COSTS)==len(ACTIVE_IDS),
    "All optimization restarts recorded":len(trial_df)==len(ACTIVE_IDS)*len(DEPTHS)*RESTARTS,
    "No locked-test optimization outcomes":locked_ids.isdisjoint(set(best_df.graph_id)),
    "All validation CX circuits correctly decomposed":all(
        row.cx_gates==2*graphs[row.graph_id].number_of_edges()*row.depth_p
        for row in resource_df.itertuples()),
    "All noise samples recorded once":len(noise_df)==expected_noise_rows and
        not noise_df.duplicated(["graph_id","depth_p","condition","repeat"]).any(),
    "All noise results use specified shot count":bool((noise_df.shots==SHOTS).all()),
    "All ratios and probabilities are valid":bool(
        best_df[["approx_ratio","p_optimal"]].ge(-1e-9).all().all() and
        best_df[["approx_ratio","p_optimal"]].le(1+1e-9).all().all() and
        noise_df[["approx_ratio","p_optimal"]].ge(-1e-9).all().all() and
        noise_df[["approx_ratio","p_optimal"]].le(1+1e-9).all().all()),
    "Zero-noise matches ideal in aggregate":noise_control_mean_abs_error<.08,
    "Paired effects use validation only":len(paired)==len(VALIDATION_IDS)*len(NOISE_CONDITIONS),
    "No locked-test noisy labels":locked_ids.isdisjoint(set(noise_df.graph_id)),
}
for label,passed in checks.items():
    print(("PASS" if passed else "FAIL")+" | "+label)
assert all(checks.values()),"A Phase 4 integrity check failed"

artifacts=[
    catalog_path,
    EXPERIMENTS / f"{PREFIX}_classical.csv", TRIAL_FILE,
    EXPERIMENTS / f"{PREFIX}_qaoa_ideal.csv",
    EXPERIMENTS / f"{PREFIX}_circuit_resources.csv",
    NOISE_RAW_PATH, EXPERIMENTS / f"{PREFIX}_noise_summary.csv",
    EXPERIMENTS / f"{PREFIX}_paired_depth_effects.csv",
    EXPERIMENTS / f"{PREFIX}_depth_benefit_bootstrap.csv",
    ideal_figure,noise_figure,
]
manifest={
    "project":"QuantumOptLab","phase":4,"config":SETTINGS,"config_hash":CONFIG_HASH,
    "versions":{"python":platform.python_version(),"qiskit":qiskit.__version__,"aer":qiskit_aer.__version__,
                "numpy":np.__version__,"scipy":__import__("scipy").__version__,"networkx":nx.__version__},
    "n_graphs_generated":len(catalog_df),"n_optimized":len(ACTIVE_IDS),
    "n_validation_noise":len(VALIDATION_IDS),"n_locked_test":len(locked_ids),
    "simulator_jobs_recorded":len(noise_df),
    "physics_max_probability_error":max_prob_error,
    "zero_noise_mean_absolute_ratio_error":noise_control_mean_abs_error,
    "integrity_checks":checks,
    "caveats":[
        "The test split remains locked: do not use it for method development.",
        "Noisy simulations use ideal-trained QAOA parameters, not noise-reoptimized parameters.",
        "CX/readout error channels are synthetic, not IBM hardware calibration.",
        "QAOA expected cut and classical best-of-restarts cut are not matching metrics.",
        "Bootstrap intervals estimate between-validation-graph variation only.",
        "No quantum advantage or generalizing learned depth selector has been established.",
    ],
    "artifacts":[str(path.relative_to(ROOT)) for path in artifacts],
    "elapsed_seconds":round(time.perf_counter()-RUN_START,2),
}
manifest_file=EXPERIMENTS / f"{PREFIX}_manifest.json"
manifest_file.write_text(json.dumps(manifest,indent=2),encoding="utf-8")
print(f"\nPASS: {sum(checks.values())}/{len(checks)} Phase 4 integrity checks")
print("Saved manifest:",manifest_file)
print("Elapsed:",manifest["elapsed_seconds"],"seconds")
print("LOCKED TEST: 20 graph structures reserved; no outcome labels generated.")
